In [1]:
%pip install scipy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
# =====================================================================
# STEP 1: IMPORT RAW DATA
# =====================================================================
df_customers = pd.read_csv('olist_customers_dataset.csv')
df_geolocation = pd.read_csv('olist_geolocation_dataset.csv')
df_order_items = pd.read_csv('olist_order_items_dataset.csv')
df_order_payments = pd.read_csv('olist_order_payments_dataset.csv')
df_order_reviews = pd.read_csv('olist_order_reviews_dataset.csv')
df_orders = pd.read_csv('olist_orders_dataset.csv')
df_products = pd.read_csv('olist_products_dataset.csv')
df_sellers = pd.read_csv('olist_sellers_dataset.csv')
df_category_translation = pd.read_csv('product_category_name_translation.csv')

In [3]:
# =====================================================================
# STEP 2: CLEANING & STANDARDIZATION
# =====================================================================
# 1. Filter delivered orders
df_orders = df_orders[df_orders['order_status'] == 'delivered']

# 2. Convert date columns
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
df_orders['order_delivered_customer_date'] = pd.to_datetime(df_orders['order_delivered_customer_date'])
df_orders['order_estimated_delivery_date'] = pd.to_datetime(df_orders['order_estimated_delivery_date'])

# 3. Fill missing text
df_products['product_category_name'] = df_products['product_category_name'].fillna('unknown')

# 4. Strip extra spaces
df_customers['customer_city'] = df_customers['customer_city'].str.strip()
df_customers['customer_state'] = df_customers['customer_state'].str.strip()

print("Data loaded & cleaned successfully!")
print("Delivered orders:", len(df_orders))

Data loaded & cleaned successfully!
Delivered orders: 96478


In [4]:
# =====================================================================
# STEP 3 & 4: MERGING & AGGREGATION (MASTER TABLE)
# =====================================================================

# 1. Join Delivered Orders with Customers (brings in customer_unique_id and region)
df_master = pd.merge(df_orders, df_customers[['customer_id', 'customer_unique_id', 'customer_state', 'customer_city']], on='customer_id', how='inner')

# 2. Aggregate total payments per order (summing up multiple payment methods if used)
order_payments_sum = df_order_payments.groupby('order_id')['payment_value'].sum().reset_index()

# 3. Join aggregated payments into our master dataset
df_master = pd.merge(df_master, order_payments_sum, on='order_id', how='left')

# 4. Aggregate average review score per order
order_reviews_avg = df_order_reviews.groupby('order_id')['review_score'].mean().reset_index()

# 5. Join review scores into our master dataset
df_master = pd.merge(df_master, order_reviews_avg, on='order_id', how='left')

print("Master dataset successfully created!")
print("Total rows (Delivered Orders):", len(df_master))
print("Total unique customers:", df_master['customer_unique_id'].nunique())

# Display first 3 rows of key columns
df_master[['order_id', 'customer_unique_id', 'payment_value', 'review_score']].head(3)


Master dataset successfully created!
Total rows (Delivered Orders): 96478
Total unique customers: 93358


,order_id,customer_unique_id,payment_value,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,38.71,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,141.46,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,179.12,5.0


In [5]:
import numpy as np

# =====================================================================
# STEP 5 & 6: DATA TRANSFORMATION & SEGMENTATION
# =====================================================================

# 1. Feature: Delivery Delay Flag (1 if actual delivery date > estimated date, else 0)
df_master['is_delayed'] = (df_master['order_delivered_customer_date'] > df_master['order_estimated_delivery_date']).astype(int)

# 2. Aggregate features at the CUSTOMER level (93,358 unique customers)
cust_df = df_master.groupby('customer_unique_id').agg(
    total_orders=('order_id', 'nunique'),
    avg_payment=('payment_value', 'mean'),
    avg_review=('review_score', 'mean'),
    was_delayed=('is_delayed', 'max') # 1 if any order was delayed
).reset_index()

# 3. Target Variable: Repeat Buyer (1) vs Churn (0)
cust_df['is_repeat'] = (cust_df['total_orders'] > 1).astype(int)

# 4. Calculate Churn Rate (Percentage of single-time buyers)
churn_rate = (cust_df['is_repeat'] == 0).mean()

# 5. Calculate Average Order Value (AOV) per segment
single_aov = cust_df[cust_df['is_repeat'] == 0]['avg_payment'].mean()
repeat_aov = cust_df[cust_df['is_repeat'] == 1]['avg_payment'].mean()

print("=== STEP 5 & 6 RESULTS ===")
print(f"Baseline Churn Rate: {churn_rate:.0%}")
print(f"Single-Buyer AOV   : R${single_aov:.2f}")
print(f"Repeat-Buyer AOV   : R${repeat_aov:.2f}")

=== STEP 5 & 6 RESULTS ===
Baseline Churn Rate: 97%
Single-Buyer AOV   : R$160.76
Repeat-Buyer AOV   : R$145.87


In [6]:
import pandas as pd
from scipy.stats import chi2_contingency

# =====================================================================
# STEP 7: HYPOTHESIS & STATISTICAL TESTING
# =====================================================================

# 1. Bin review scores into Low (<= 3) vs High (> 3)
cust_df['low_review'] = (cust_df['avg_review'] <= 3).astype(int)

# 2. Build Contingency Tables
contingency_review = pd.crosstab(cust_df['is_repeat'], cust_df['low_review'])
contingency_delay = pd.crosstab(cust_df['is_repeat'], cust_df['was_delayed'])

# 3. Calculate Chi-Square Significance (p-values)
_, p_review, _, _ = chi2_contingency(contingency_review)
_, p_delay, _, _ = chi2_contingency(contingency_delay)

print("=== STEP 7 STATISTICAL TEST RESULTS ===")
print(f"Review Score vs Churn p-value  : {p_review:.4f}")
print(f"Delivery Delay vs Churn p-value: {p_delay:.4f}")


=== STEP 7 STATISTICAL TEST RESULTS ===
Review Score vs Churn p-value  : 0.0062
Delivery Delay vs Churn p-value: 0.0000


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# =====================================================================
# STEP 8: PREDICTIVE MODELING (LOGISTIC REGRESSION)
# =====================================================================

# 1. Fill missing values (NaNs) in features before modeling
cust_df['avg_review'] = cust_df['avg_review'].fillna(cust_df['avg_review'].mean())
cust_df['avg_payment'] = cust_df['avg_payment'].fillna(0)

# 2. Define features (X) and target variable (y)
X = cust_df[['avg_payment', 'avg_review', 'was_delayed']]
y = cust_df['is_repeat']

# 3. Split data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Train Logistic Regression Model
model = LogisticRegression()
model.fit(X_train, y_train)

# 5. Predict probabilities and calculate ROC-AUC score
y_pred_proba = model.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, y_pred_proba)

print("=== STEP 8 MODEL RESULTS ===")
print(f"Logistic Regression ROC-AUC Score: {auc_score:.4f}")

=== STEP 8 MODEL RESULTS ===
Logistic Regression ROC-AUC Score: 0.5156


In [10]:
# =====================================================================
#EXPORT DATASETS 
# =====================================================================

#Export directly to Excel (.xlsx)
df_master.to_excel('olist_master_orders_processed.xlsx', index=False)
cust_df.to_excel('olist_customer_metrics.xlsx', index=False)

print("=== EXCEL EXPORT COMPLETE ===")
print("  - olist_master_orders_processed.xlsx (Order Level)")
print("  - olist_customer_metrics.xlsx        (Customer Level)")

=== EXCEL EXPORT COMPLETE ===
  - olist_master_orders_processed.xlsx (Order Level)
  - olist_customer_metrics.xlsx        (Customer Level)


In [2]:
import os
import sqlite3
import pandas as pd

# =====================================================================
# 1. DATABASE INITIALIZATION & DATA LOADING
# =====================================================================
conn = sqlite3.connect('olist_analytics.db')

# Load files from disk (checking for .xlsx or .csv)
def load_data():
    global df_master, cust_df
    
    # Try reading Excel first
    if os.path.exists('olist_master_orders_processed.xlsx'):
        df_master = pd.read_excel('olist_master_orders_processed.xlsx')
    elif os.path.exists('olist_master_orders_processed.csv'):
        df_master = pd.read_csv('olist_master_orders_processed.csv')
    else:
        raise FileNotFoundError("Could not find olist_master_orders_processed file (.xlsx or .csv).")

    if os.path.exists('olist_customer_metrics.xlsx'):
        cust_df = pd.read_excel('olist_customer_metrics.xlsx')
    elif os.path.exists('olist_customer_metrics.csv'):
        cust_df = pd.read_csv('olist_customer_metrics.csv')
    else:
        raise FileNotFoundError("Could not find olist_customer_metrics file (.xlsx or .csv).")

# Execute load if dataframes are not in local memory
try:
    df_master
    cust_df
except NameError:
    load_data()

# Write into SQLite relational tables
df_master.to_sql('master_orders', conn, if_exists='replace', index=False)
cust_df.to_sql('customer_metrics', conn, if_exists='replace', index=False)

print(" SQL ENVIRONMENT ESTABLISHED SUCCESSFULLY ")
print("Tables 'master_orders' and 'customer_metrics' are online.\n")

# Detect columns automatically to prevent schema mismatches
master_cols = [col[1] for col in conn.execute("PRAGMA table_info(master_orders);").fetchall()]
cust_cols = [col[1] for col in conn.execute("PRAGMA table_info(customer_metrics);").fetchall()]

state_col_master = next((c for c in master_cols if 'state' in c.lower()), None)
state_col_cust = next((c for c in cust_cols if 'state' in c.lower()), None)

if state_col_master:
    state_ref = f"m.{state_col_master}"
elif state_col_cust:
    state_ref = f"c.{state_col_cust}"
else:
    state_ref = "m.customer_state"

print("-" * 80 + "\n")


# =====================================================================
# QUERY 1: RFM DECILE SEGMENTATION & REVENUE CONCENTRATION (NTILE)
# =====================================================================
query_rfm_deciles = """
with customer_rfm as (
    select 
        customer_unique_id,
        total_orders as Frequency,
        (avg_payment * total_orders) as Monetary
    from customer_metrics
),
ranked_deciles as (
    select 
        customer_unique_id,
        Frequency,
        Monetary,
        ntile(10) over (order by Monetary asc) as Monetary_Decile
    from customer_rfm
)
select 
    case 
        when Monetary_Decile >= 9 then 'Top 20% (Deciles 9-10)'
        else 'Bottom 80% (Deciles 1-8)'
    end as Customer_Segment,
    count(customer_unique_id) as Customer_Count,
    round(sum(Monetary), 2) as Total_Revenue,
    round(100.0 * sum(Monetary) / (select sum(Monetary) from customer_rfm), 2) as [Revenue Share %]
from ranked_deciles
group by Customer_Segment
order by Total_Revenue desc;
"""

print("=== SQL QUERY 1: RFM DECILE REVENUE CONCENTRATION ===")
df_sql_result1 = pd.read_sql_query(query_rfm_deciles, conn)
print(df_sql_result1)
print("\n" + "-" * 80 + "\n")


# =====================================================================
# QUERY 2: TOP REVENUE CUSTOMERS PER STATE (DENSE_RANK PARTITION)
# =====================================================================
query_top_state_customers = f"""
with customer_spend as (
    select 
        {state_ref} as State,
        m.customer_unique_id,
        round(sum(m.payment_value), 2) as Total_Spent,
        count(distinct m.order_id) as Total_Orders
    from master_orders m
    join customer_metrics c on m.customer_unique_id = c.customer_unique_id
    group by State, m.customer_unique_id
),
ranked_customers as (
    select 
        State,
        customer_unique_id,
        Total_Spent,
        Total_Orders,
        dense_rank() over (partition by State order by Total_Spent desc) as State_Rank
    from customer_spend
)
select 
    State,
    customer_unique_id as Customer_ID,
    Total_Spent as [Total Spend (R$)],
    Total_Orders as [Order Count]
from ranked_customers
where State_Rank <= 2
order by State, State_Rank;
"""

print("=== SQL QUERY 2: TOP 2 SPENDERS PER STATE (PARTITION BY STATE) ===")
df_sql_result2 = pd.read_sql_query(query_top_state_customers, conn)
print(df_sql_result2.head(10))
print("\n" + "-" * 80 + "\n")


# =====================================================================
# QUERY 3: DELIVERY DELAY vs CHURN CONVERGENCE (RELATIONAL JOIN)
# =====================================================================
query_delay_churn = f"""
select 
    {state_ref} as State,
    count(distinct m.order_id) as Total_Orders,
    round(avg(case when m.order_delivered_customer_date > m.order_estimated_delivery_date then 1.0 else 0.0 end) * 100, 2) as [Delay Rate %],
    round(avg(m.review_score), 2) as [Avg Review Score],
    round(avg(c.is_repeat) * 100, 2) as [Repeat Customer Rate %]
from master_orders m
join customer_metrics c on m.customer_unique_id = c.customer_unique_id
group by State
having count(distinct m.order_id) >= 500
order by [Delay Rate %] desc;
"""

print("=== SQL QUERY 3: STATE-LEVEL DELIVERY DELAY vs REPEAT RATE ===")
df_sql_result3 = pd.read_sql_query(query_delay_churn, conn)
print(df_sql_result3.head(8))
print("\n" + "-" * 80 + "\n")


# =====================================================================
# QUERY 4: HIGH-VALUE REPEAT CHAMPIONS (FILTRATION & SELECTION)
# =====================================================================
query_repeat_champions = """
select 
    customer_unique_id as Customer_ID,
    total_orders as [Total Orders],
    round(avg_payment, 2) as [Avg Order Value (R$)],
    round(avg_payment * total_orders, 2) as [Total Lifetime Spend (R$)],
    round(avg_review, 1) as [Avg Review Score]
from customer_metrics
where is_repeat = 1 
  and avg_review >= 4.0 
  and was_delayed = 0
order by [Total Lifetime Spend (R$)] desc
limit 10;
"""

print("=== SQL QUERY 4: TOP REPEAT CHAMPIONS (HIGH SPEND & HIGH SATISFACTION) ===")
df_sql_result4 = pd.read_sql_query(query_repeat_champions, conn)
print(df_sql_result4)

 SQL ENVIRONMENT ESTABLISHED SUCCESSFULLY 
Tables 'master_orders' and 'customer_metrics' are online.

--------------------------------------------------------------------------------

=== SQL QUERY 1: RFM DECILE REVENUE CONCENTRATION ===
           Customer_Segment  Customer_Count  Total_Revenue  Revenue Share %
0    Top 20% (Deciles 9-10)           18670     8254457.04            53.52
1  Bottom 80% (Deciles 1-8)           74688     7168004.73            46.48

--------------------------------------------------------------------------------

=== SQL QUERY 2: TOP 2 SPENDERS PER STATE (PARTITION BY STATE) ===
  State                       Customer_ID  Total Spend (R$)  Order Count
0    AC  62a459e5629b03dd73134964df732077           1251.70            1
1    AC  086d6b5b5ba195a91aa0a6ec8e75d1a4            995.18            1
2    AL  03adb96012772d4374285b6a701f3da9           2269.98            1
3    AL  9b661514ad84d0489ac73d9e24cbf169           1942.00            1
4    AM  23841d4abf